# Módulo 03 · Aula 03 — Joins e Subconsultas

> **Manual de Estudos Interativo** · Trilha Engenharia de Software & Dados
> Projeto transversal: **Atlas / Aurora Comércio**

Na aula 01 você **separou** os dados em tabelas para eliminar redundância. Agora vai **juntá-los de volta** — só que sob demanda, e sem duplicar nada no disco.

Essa é a beleza do modelo relacional: você guarda normalizado e monta a visão que precisar na hora da consulta.

## O que você vai aprender aqui

| # | Tópico | Pergunta que responde |
|---|--------|------------------------|
| 1 | `INNER JOIN` | "Quais linhas existem nas **duas** tabelas?" |
| 2 | `LEFT JOIN` | "Todos os produtos, **mesmo os que nunca venderam**" |
| 3 | `RIGHT` / `FULL` | E o que fazer no SQLite |
| 4 | Anti-join | "Quem **não** tem correspondência?" |
| 5 | `CROSS JOIN` | Produto cartesiano (útil e perigoso) |
| 6 | Self join | "Compare a tabela com ela mesma" |
| 7 | Subconsultas | Escalares, de lista, correlacionadas |
| 8 | `EXISTS` | O filtro mais eficiente para "existe pelo menos um" |
| 9 | **CTEs (`WITH`)** | Como escrever SQL que outra pessoa consegue ler |
| 10 | `UNION` e conjuntos | Empilhar resultados |

## ⚙️ Preparando o banco

Mesma base da aula anterior: 6 categorias, 20 produtos, 30 clientes, 180 pedidos, ~340 itens. Semente fixa, então os números batem com os do manual.

> ▶️ **Execute esta célula primeiro.**

In [ ]:
import sqlite3
import random

# ═══════════════════════════════════════════════════════════════
#  Banco de treino da Aurora Comércio
#  Execute esta célula UMA VEZ, antes de qualquer outra.
#  Ela é idempotente: pode rodar de novo a qualquer momento.
# ═══════════════════════════════════════════════════════════════

CONN = sqlite3.connect(":memory:")
CONN.execute("PRAGMA foreign_keys = ON")

CONN.executescript("""
CREATE TABLE categorias (
    id           INTEGER PRIMARY KEY,
    nome         TEXT    NOT NULL UNIQUE,
    margem_alvo  REAL    NOT NULL DEFAULT 0.25 CHECK (margem_alvo BETWEEN 0 AND 1)
);

CREATE TABLE produtos (
    id           INTEGER PRIMARY KEY,
    sku          TEXT    NOT NULL UNIQUE,
    nome         TEXT    NOT NULL,
    categoria_id INTEGER NOT NULL REFERENCES categorias(id) ON DELETE RESTRICT,
    preco        REAL    NOT NULL CHECK (preco >= 0),
    custo        REAL    NOT NULL CHECK (custo >= 0),
    estoque      INTEGER NOT NULL DEFAULT 0 CHECK (estoque >= 0),
    ativo        INTEGER NOT NULL DEFAULT 1 CHECK (ativo IN (0,1))
);

CREATE TABLE clientes (
    id            INTEGER PRIMARY KEY,
    nome          TEXT NOT NULL,
    email         TEXT NOT NULL UNIQUE,
    cidade        TEXT NOT NULL,
    uf            TEXT NOT NULL CHECK (length(uf) = 2),
    segmento      TEXT NOT NULL DEFAULT 'varejo'
                       CHECK (segmento IN ('varejo','corporativo')),
    data_cadastro TEXT NOT NULL,
    telefone      TEXT
);

CREATE TABLE pedidos (
    id          INTEGER PRIMARY KEY,
    cliente_id  INTEGER NOT NULL REFERENCES clientes(id) ON DELETE RESTRICT,
    data_pedido TEXT    NOT NULL,
    status      TEXT    NOT NULL CHECK (status IN ('pago','pendente','cancelado')),
    canal       TEXT    NOT NULL CHECK (canal IN ('site','app','marketplace')),
    frete       REAL    NOT NULL DEFAULT 0 CHECK (frete >= 0)
);

CREATE TABLE itens_pedido (
    id             INTEGER PRIMARY KEY,
    pedido_id      INTEGER NOT NULL REFERENCES pedidos(id)  ON DELETE CASCADE,
    produto_id     INTEGER NOT NULL REFERENCES produtos(id) ON DELETE RESTRICT,
    quantidade     INTEGER NOT NULL CHECK (quantidade > 0),
    preco_unitario REAL    NOT NULL CHECK (preco_unitario >= 0),
    UNIQUE (pedido_id, produto_id)
);
""")

_CATEGORIAS = [(1, "Notebooks", 0.18), (2, "Monitores", 0.22), (3, "Periféricos", 0.38),
               (4, "Armazenamento", 0.30), (5, "Redes", 0.28), (6, "Áudio", 0.35)]

_PRODUTOS = [
    ("NB-DELL-15",  "Notebook Dell Inspiron 15",     1, 2599.90, 2120.00,  14),
    ("NB-ACER-N5",  "Notebook Acer Nitro 5",         1, 3299.00, 2780.00,   7),
    ("NB-LEN-IP3",  "Notebook Lenovo IdeaPad 3",     1, 2199.00, 1850.00,  22),
    ("NB-APPL-M2",  "MacBook Air M2",                1, 9499.00, 8300.00,   3),
    ("MO-LG-24UW",  "Monitor LG 24 UltraWide",       2, 1199.00,  920.00,  31),
    ("MO-SAM-ODY",  "Monitor Samsung Odyssey 27",    2, 1849.00, 1420.00,  12),
    ("MO-AOC-22",   "Monitor AOC 22 Full HD",        2,  749.00,  560.00,  45),
    ("PE-LOG-MX3",  "Mouse Logitech MX Master 3",    3,  549.00,  340.00,  88),
    ("PE-LOG-M170", "Mouse Logitech M170",           3,   89.90,   52.00, 240),
    ("PE-RED-K552", "Teclado Redragon K552",         3,  249.00,  150.00,  64),
    ("PE-LOG-C920", "Webcam Logitech C920",          3,  449.00,  290.00,  37),
    ("AU-HYP-CL2",  "Headset HyperX Cloud II",       6,  399.00,  255.00,  29),
    ("AU-JBL-T510", "Fone JBL Tune 510BT",           6,  229.00,  140.00,  73),
    ("AR-SSD-1TB",  "SSD NVMe 1TB Kingston",         4,  489.00,  360.00,  52),
    ("AR-SSD-480",  "SSD SATA 480GB Sandisk",        4,  229.00,  158.00,  96),
    ("AR-HD-2TB",   "HD Externo 2TB Seagate",        4,  549.00,  410.00,  18),
    ("AR-PEN-128",  "Pendrive 128GB Sandisk",        4,   79.90,   44.00, 180),
    ("RE-TPL-AX55", "Roteador TP-Link Archer AX55",  5,  699.00,  505.00,  26),
    ("RE-TPL-RE30", "Repetidor TP-Link RE305",       5,  229.00,  152.00,  41),
    ("RE-INT-AX20", "Placa de Rede Intel AX200",     5,  189.00,  124.00,  33),
]

_NOMES = ["Ana Costa", "Bruno Rocha", "Carla Dias", "Daniel Souza", "Elisa Martins",
          "Fábio Nunes", "Gustavo Reis", "Helena Prado", "Igor Batista", "Julia Andrade",
          "Lucas Moreira", "Maria Souza", "Nathalia Freitas", "Otávio Pinto",
          "Priscila Gomes", "Rafael Torres", "Sabrina Melo", "Thiago Barros",
          "Vanessa Lima", "William Cruz", "Beatriz Almeida", "Caio Ferreira",
          "Débora Ramos", "Eduardo Pires", "Fernanda Vieira", "Gabriel Mendes",
          "Isabela Rocha", "João Lima", "Karina Duarte", "Leonardo Castro"]

_CIDADES = [("Campinas", "SP"), ("São Paulo", "SP"), ("Sorocaba", "SP"),
            ("Ribeirão Preto", "SP"), ("Jundiaí", "SP"), ("Santos", "SP"),
            ("Belo Horizonte", "MG"), ("Uberlândia", "MG"), ("Curitiba", "PR"),
            ("Londrina", "PR"), ("Porto Alegre", "RS"), ("Florianópolis", "SC"),
            ("Rio de Janeiro", "RJ"), ("Niterói", "RJ"), ("Salvador", "BA"),
            ("Recife", "PE"), ("Fortaleza", "CE"), ("Brasília", "DF"),
            ("Goiânia", "GO"), ("Vitória", "ES")]

_rnd = random.Random(42)     # semente fixa = todos veem os mesmos números

CONN.executemany("INSERT INTO categorias VALUES (?,?,?)", _CATEGORIAS)
CONN.executemany(
    "INSERT INTO produtos (sku,nome,categoria_id,preco,custo,estoque,ativo) "
    "VALUES (?,?,?,?,?,?,1)", _PRODUTOS)

_acentos = str.maketrans("áéíóúãõâêôç", "aeiouaoaeoc")
_clientes = []
for _i, _nome in enumerate(_NOMES, 1):
    _cidade, _uf = _rnd.choice(_CIDADES)
    _login = _nome.split()[0].lower().translate(_acentos)
    _clientes.append((
        _i, _nome, f"{_login}{_i}@email.com", _cidade, _uf,
        "corporativo" if _rnd.random() < 0.25 else "varejo",
        f"2026-{_rnd.randint(1, 6):02d}-{_rnd.randint(1, 28):02d}",
        f"(19) 9{_rnd.randint(1000, 9999)}-{_rnd.randint(1000, 9999)}" if _rnd.random() < 0.6 else None,
    ))
CONN.executemany("INSERT INTO clientes VALUES (?,?,?,?,?,?,?,?)", _clientes)

_pedidos, _itens, _id_item = [], [], 0
# Os 3 últimos clientes ficam SEM pedido de propósito: toda base real tem
# gente que se cadastrou e nunca comprou, e você precisa saber encontrá-los.
for _pid in range(1, 181):
    _mes = _rnd.choices([5, 6, 7], weights=[2, 3, 4])[0]
    _pedidos.append((
        _pid, _rnd.randint(1, len(_NOMES) - 3),
        f"2026-{_mes:02d}-{_rnd.randint(1, 28):02d}",
        _rnd.choices(["pago", "pendente", "cancelado"], weights=[80, 12, 8])[0],
        _rnd.choices(["site", "app", "marketplace"], weights=[50, 30, 20])[0],
        _rnd.choice([0.0, 9.90, 19.90, 29.90]),
    ))
    _n_itens = _rnd.choices([1, 2, 3, 4], weights=[45, 30, 17, 8])[0]
    for _prod in _rnd.sample(range(1, len(_PRODUTOS) + 1), _n_itens):
        _id_item += 1
        _itens.append((
            _id_item, _pid, _prod,
            _rnd.choices([1, 2, 3, 5, 10], weights=[55, 22, 12, 7, 4])[0],
            round(_PRODUTOS[_prod - 1][3] * _rnd.choice([1.0, 1.0, 1.0, 0.95, 0.90]), 2),
        ))
CONN.executemany("INSERT INTO pedidos VALUES (?,?,?,?,?,?)", _pedidos)
CONN.executemany("INSERT INTO itens_pedido VALUES (?,?,?,?,?)", _itens)
CONN.commit()


# ── Funções auxiliares ───────────────────────────────────────
def _fmt(valor):
    if valor is None:
        return "NULL"
    if isinstance(valor, float):
        return f"{valor:,.2f}"
    if isinstance(valor, int):
        return f"{valor:,}"
    return str(valor)


def sql(consulta, parametros=(), limite=30):
    """Executa uma consulta e imprime o resultado formatado."""
    try:
        cursor = CONN.execute(consulta, parametros)
    except sqlite3.Error as erro:
        print(f"❌ {type(erro).__name__}: {erro}")
        return None

    if cursor.description is None:
        CONN.commit()
        print(f"✅ OK — {cursor.rowcount} linha(s) afetada(s)" if cursor.rowcount >= 0 else "✅ OK")
        return None

    colunas = [d[0] for d in cursor.description]
    linhas = cursor.fetchall()
    total = len(linhas)
    linhas = linhas[:limite]
    if not linhas:
        print("(nenhuma linha)")
        return []

    texto = [[_fmt(v) for v in linha] for linha in linhas]
    numerica = [
        any(isinstance(l[i], (int, float)) for l in linhas)
        and all(isinstance(l[i], (int, float)) or l[i] is None for l in linhas)
        for i in range(len(colunas))
    ]
    larguras = [max(len(colunas[i]), max(len(l[i]) for l in texto))
                for i in range(len(colunas))]

    def borda(e, m, d):
        return e + m.join("─" * (w + 2) for w in larguras) + d

    print(borda("┌", "┬", "┐"))
    print("│ " + " │ ".join(c.ljust(w) for c, w in zip(colunas, larguras)) + " │")
    print(borda("├", "┼", "┤"))
    for linha in texto:
        print("│ " + " │ ".join(
            (v.rjust(w) if numerica[i] else v.ljust(w))
            for i, (v, w) in enumerate(zip(linha, larguras))) + " │")
    print(borda("└", "┴", "┘"))
    print(f"{total} linha(s)" + (f" — exibindo as {limite} primeiras" if total > limite else ""))
    return linhas


def ddl(script):
    """Executa um script com vários comandos."""
    try:
        CONN.executescript(script)
        CONN.commit()
        print("✅ Script executado")
    except sqlite3.Error as erro:
        print(f"❌ {type(erro).__name__}: {erro}")


print("✅ Banco da Aurora criado em memória\n")
sql("""
SELECT 'categorias'   AS tabela, COUNT(*) AS linhas FROM categorias
UNION ALL SELECT 'produtos',     COUNT(*) FROM produtos
UNION ALL SELECT 'clientes',     COUNT(*) FROM clientes
UNION ALL SELECT 'pedidos',      COUNT(*) FROM pedidos
UNION ALL SELECT 'itens_pedido', COUNT(*) FROM itens_pedido
""")

## 1. O problema que o JOIN resolve

A tabela `pedidos` guarda `cliente_id`, não o nome do cliente. Isso é correto — o nome mora em `clientes`, uma vez só. Mas o relatório precisa do nome.

In [ ]:
sql("SELECT id, cliente_id, data_pedido, status, canal FROM pedidos LIMIT 5")

### A sintaxe

```sql
SELECT colunas
FROM tabela_a
JOIN tabela_b ON tabela_a.coluna = tabela_b.coluna;
```

A condição depois do `ON` é o **critério de correspondência**. Quase sempre é `chave_estrangeira = chave_primária`.

In [ ]:
sql("""
SELECT
    p.id           AS pedido,
    p.data_pedido,
    c.nome         AS cliente,
    c.cidade,
    p.status
FROM pedidos p
JOIN clientes c ON c.id = p.cliente_id
ORDER BY p.id
LIMIT 8
""")

> 💡 **Sempre use apelidos de tabela** (`pedidos p`, `clientes c`). Eles deixam a consulta mais curta e, quando duas tabelas têm colunas de mesmo nome (`id`, `nome`), tornam-se obrigatórios. Escolha letras que lembrem a tabela — `p`, `c`, `i`, `pr`, `cat` — não `a`, `b`, `t1`.

## 2. `INNER JOIN` — só o que casa dos dois lados

`JOIN` sozinho significa `INNER JOIN`. Ele devolve **apenas as linhas que têm correspondência nas duas tabelas**.

```
   A          B              INNER JOIN
 ┌─────┐  ┌─────┐          ┌─────┐
 │  ███│██│███  │          │  ███│
 │  ███│██│███  │    ->    │  ███│      só a interseção
 └─────┘  └─────┘          └─────┘
```

In [ ]:
# Para demonstrar, criamos um produto que nunca foi vendido
sql("""INSERT INTO produtos (sku, nome, categoria_id, preco, custo, estoque)
       VALUES ('AC-SUP-NB1', 'Suporte para Notebook', 3, 129.00, 78.00, 50)""")

sql("SELECT COUNT(*) AS total_produtos FROM produtos")

In [ ]:
# INNER JOIN: o produto novo NÃO aparece, porque não tem item de pedido
sql("""
SELECT
    pr.sku,
    pr.nome,
    COUNT(i.id)        AS vezes_vendido,
    SUM(i.quantidade)  AS unidades
FROM produtos pr
JOIN itens_pedido i ON i.produto_id = pr.id
GROUP BY pr.id, pr.sku, pr.nome
ORDER BY unidades DESC
""")

### Juntando várias tabelas

Você pode encadear quantos `JOIN` quiser. Cada um adiciona uma tabela ao conjunto.

O caminho pelo modelo da Aurora é: `itens_pedido → pedidos → clientes` e `itens_pedido → produtos → categorias`.

In [ ]:
sql("""
SELECT
    p.id                                       AS pedido,
    p.data_pedido,
    c.nome                                     AS cliente,
    c.cidade,
    pr.nome                                    AS produto,
    cat.nome                                   AS categoria,
    i.quantidade                               AS qtd,
    i.preco_unitario,
    ROUND(i.quantidade * i.preco_unitario, 2)  AS total_item
FROM itens_pedido i
JOIN pedidos    p   ON p.id   = i.pedido_id
JOIN clientes   c   ON c.id   = p.cliente_id
JOIN produtos   pr  ON pr.id  = i.produto_id
JOIN categorias cat ON cat.id = pr.categoria_id
WHERE p.status = 'pago'
ORDER BY total_item DESC
LIMIT 10
""")

### ⚠️ O JOIN multiplica linhas

Este é o mal-entendido mais custoso do SQL.

Quando você junta `pedidos` com `itens_pedido`, **cada pedido aparece uma vez para cada item que ele tem**. Um pedido com 3 itens vira 3 linhas.

Consequência: `COUNT(*)` conta **itens**, não pedidos. E `SUM(p.frete)` soma o frete **3 vezes**.

In [ ]:
sql("""
SELECT
    COUNT(*)                        AS linhas_apos_join,
    COUNT(DISTINCT p.id)            AS pedidos_reais,
    (SELECT COUNT(*) FROM pedidos)  AS pedidos_na_tabela
FROM pedidos p
JOIN itens_pedido i ON i.pedido_id = p.id
""")

In [ ]:
# 🔴 O erro clássico: somar o frete depois de juntar com os itens
sql("""
SELECT
    ROUND((SELECT SUM(frete) FROM pedidos), 2) AS frete_correto,
    ROUND(SUM(p.frete), 2)                     AS frete_inflado_pelo_join
FROM pedidos p
JOIN itens_pedido i ON i.pedido_id = p.id
""")

> 🔴 **Como se proteger:**
>
> 1. Use `COUNT(DISTINCT id)` quando quiser contar entidades, não linhas.
> 2. Para somar valores da tabela "pai" (frete, desconto do pedido), **agregue antes de juntar** — com uma subconsulta ou CTE.
> 3. Desconfie sempre que um total ficar suspeitosamente alto. Compare com a contagem sem `JOIN`.

## 3. `LEFT JOIN` — tudo da esquerda, casando o que der

Devolve **todas** as linhas da tabela da esquerda. Onde não há correspondência à direita, preenche com `NULL`.

```
   A          B              LEFT JOIN
 ┌─────┐  ┌─────┐          ┌─────┐
 │█████│██│███  │          │█████│██
 │█████│██│███  │    ->    │█████│██   tudo de A
 └─────┘  └─────┘          └─────┘
```

**É o que você quer sempre que a resposta precisa incluir os "zeros".**

In [ ]:
# Agora o produto sem vendas APARECE, com NULL nas colunas de venda
sql("""
SELECT
    pr.sku,
    pr.nome,
    COUNT(i.id)       AS vezes_vendido,
    SUM(i.quantidade) AS unidades
FROM produtos pr
LEFT JOIN itens_pedido i ON i.produto_id = pr.id
GROUP BY pr.id, pr.sku, pr.nome
ORDER BY unidades
LIMIT 6
""")

In [ ]:
# COALESCE transforma os NULL em zeros apresentáveis
sql("""
SELECT
    pr.sku,
    pr.nome,
    COUNT(i.id)                       AS vezes_vendido,
    COALESCE(SUM(i.quantidade), 0)    AS unidades,
    ROUND(COALESCE(SUM(i.quantidade * i.preco_unitario), 0), 2) AS receita
FROM produtos pr
LEFT JOIN itens_pedido i ON i.produto_id = pr.id
GROUP BY pr.id, pr.sku, pr.nome
ORDER BY receita
LIMIT 6
""")

> 💡 **Repare em `COUNT(i.id)` e não `COUNT(*)`.** Com `LEFT JOIN`, a linha do produto sem vendas existe — então `COUNT(*)` devolveria **1**, não 0. Já `COUNT(i.id)` conta valores não nulos e devolve corretamente **0**.
>
> Esse detalhe já produziu muito relatório onde "todo produto vendeu pelo menos uma vez".

### ⚠️ `WHERE` vs `ON` no `LEFT JOIN`

Colocar uma condição da tabela da direita no `WHERE` **anula o `LEFT JOIN`**, transformando-o silenciosamente em `INNER JOIN`.

Motivo: o `LEFT JOIN` gera `NULL` nas colunas da direita; o `WHERE` então testa `NULL = 'pago'`, que não é verdadeiro, e a linha é descartada.

**A regra:** condição sobre a tabela da **direita** vai no `ON`. Condição sobre a **esquerda** vai no `WHERE`.

In [ ]:
print("① Condição no WHERE — vira INNER JOIN sem avisar:")
sql("""
SELECT COUNT(*) AS produtos_no_resultado
FROM produtos pr
LEFT JOIN itens_pedido i ON i.produto_id = pr.id
LEFT JOIN pedidos p      ON p.id = i.pedido_id
WHERE p.status = 'pago'
""")

print("\n② Condição no ON — o LEFT JOIN se mantém:")
sql("""
SELECT COUNT(DISTINCT pr.id) AS produtos_no_resultado
FROM produtos pr
LEFT JOIN itens_pedido i ON i.produto_id = pr.id
LEFT JOIN pedidos p      ON p.id = i.pedido_id AND p.status = 'pago'
""")

print("\n③ Total de produtos na tabela:")
sql("SELECT COUNT(*) AS total FROM produtos")

## 4. Anti-join — encontrando quem **não** tem correspondência

`LEFT JOIN` + `WHERE direita.chave IS NULL` = "me dê os órfãos".

É como você responde perguntas do tipo:

- Quais produtos nunca venderam?
- Quais clientes se cadastraram e nunca compraram?
- Quais categorias estão sem produto ativo?

In [ ]:
sql("""
SELECT
    pr.sku,
    pr.nome,
    pr.preco,
    pr.estoque,
    ROUND(pr.preco * pr.estoque, 2) AS capital_parado
FROM produtos pr
LEFT JOIN itens_pedido i ON i.produto_id = pr.id
WHERE i.id IS NULL
ORDER BY capital_parado DESC
""")

In [ ]:
# Clientes que nunca compraram
sql("""
SELECT
    c.id,
    c.nome,
    c.cidade,
    c.uf,
    c.data_cadastro
FROM clientes c
LEFT JOIN pedidos p ON p.cliente_id = c.id
WHERE p.id IS NULL
ORDER BY c.data_cadastro
""")

## 5. `RIGHT JOIN` e `FULL OUTER JOIN`

- **`RIGHT JOIN`** — o espelho do `LEFT`: tudo da direita.
- **`FULL OUTER JOIN`** — tudo dos dois lados, com `NULL` onde não casar.

> ℹ️ **Suporte no SQLite:** ambos existem a partir da **versão 3.39** (2022). Em versões anteriores, só há `LEFT JOIN`.
>
> Na prática, isso quase não importa: **`RIGHT JOIN` nunca é necessário** — basta inverter a ordem das tabelas e usar `LEFT`. E é mais legível assim, porque a tabela "principal" fica visualmente à esquerda.

In [ ]:
print("Versão do SQLite:", CONN.execute("SELECT sqlite_version()").fetchone()[0])

# Estes dois são equivalentes:
print("\n① LEFT JOIN (recomendado):")
sql("""
SELECT cat.nome AS categoria, COUNT(pr.id) AS produtos
FROM categorias cat
LEFT JOIN produtos pr ON pr.categoria_id = cat.id
GROUP BY cat.id, cat.nome
ORDER BY produtos DESC
""")

In [ ]:
# FULL OUTER JOIN simulado com UNION — funciona em qualquer versão
sql("""
SELECT cat.nome AS categoria, pr.sku
FROM categorias cat
LEFT JOIN produtos pr ON pr.categoria_id = cat.id

UNION

SELECT cat.nome, pr.sku
FROM produtos pr
LEFT JOIN categorias cat ON cat.id = pr.categoria_id

ORDER BY categoria, sku
LIMIT 10
""")

## 6. `CROSS JOIN` — o produto cartesiano

Combina **cada** linha da esquerda com **cada** linha da direita. 20 produtos × 30 clientes = 600 linhas.

> 🔴 **Cuidado:** se você escrever `FROM a, b` e **esquecer o `WHERE`**, obtém um cross join acidental. Com duas tabelas de 100 mil linhas, isso são 10 bilhões de linhas e o banco trava. É a causa nº 1 de "a consulta não termina nunca".

**Onde ele é legítimo:** gerar todas as combinações possíveis — por exemplo, uma grade completa mês × categoria para um relatório que precisa mostrar zeros.

In [ ]:
sql("""
SELECT COUNT(*) AS combinacoes
FROM produtos
CROSS JOIN clientes
""")

In [ ]:
# Uso legítimo: grade completa mês × categoria, sem buracos
sql("""
WITH meses(mes) AS (
    VALUES ('2026-05'), ('2026-06'), ('2026-07')
)
SELECT
    m.mes,
    cat.nome AS categoria,
    ROUND(COALESCE(SUM(i.quantidade * i.preco_unitario), 0), 2) AS faturamento
FROM meses m
CROSS JOIN categorias cat
LEFT JOIN produtos pr     ON pr.categoria_id = cat.id
LEFT JOIN itens_pedido i  ON i.produto_id = pr.id
LEFT JOIN pedidos p       ON p.id = i.pedido_id
                          AND p.status = 'pago'
                          AND strftime('%Y-%m', p.data_pedido) = m.mes
GROUP BY m.mes, cat.id, cat.nome
ORDER BY m.mes, faturamento DESC
LIMIT 18
""")

## 7. Self join — a tabela consigo mesma

Junte uma tabela com ela própria usando **dois apelidos diferentes**. Serve para comparar linhas entre si.

In [ ]:
# Pares de clientes da mesma cidade
sql("""
SELECT
    a.cidade,
    a.nome AS cliente_1,
    b.nome AS cliente_2
FROM clientes a
JOIN clientes b ON b.cidade = a.cidade
                AND b.id > a.id       -- evita repetir o par e o auto-par
ORDER BY a.cidade, a.nome
LIMIT 12
""")

> 💡 **A condição `b.id > a.id`** faz duas coisas de uma vez: evita casar a linha com ela mesma (`a.id = b.id`) e evita trazer o par duas vezes (A-B e B-A). É o idioma padrão para "combinações de dois a dois".

In [ ]:
# Produtos da mesma categoria com preço próximo — candidatos a canibalização
sql("""
SELECT
    cat.nome                             AS categoria,
    a.nome                               AS produto_a,
    a.preco                              AS preco_a,
    b.nome                               AS produto_b,
    b.preco                              AS preco_b,
    ROUND(ABS(a.preco - b.preco), 2)     AS diferenca
FROM produtos a
JOIN produtos b     ON b.categoria_id = a.categoria_id AND b.id > a.id
JOIN categorias cat ON cat.id = a.categoria_id
WHERE ABS(a.preco - b.preco) < 100
ORDER BY diferenca
""")

## 8. Subconsultas

Uma consulta dentro de outra. Existem três tipos, e a diferença importa.

| Tipo | Devolve | Onde aparece |
|------|---------|--------------|
| **Escalar** | Um único valor | Em qualquer lugar onde caberia um valor |
| **De lista** | Uma coluna, várias linhas | Com `IN`, `ANY`, `ALL` |
| **Derivada** | Uma tabela inteira | No `FROM` |
| **Correlacionada** | Depende da linha externa | No `WHERE` ou `SELECT` |

In [ ]:
# Escalar: produtos acima da média de preço
sql("""
SELECT
    sku,
    nome,
    preco,
    (SELECT ROUND(AVG(preco), 2) FROM produtos) AS media_geral,
    ROUND(preco - (SELECT AVG(preco) FROM produtos), 2) AS diferenca
FROM produtos
WHERE preco > (SELECT AVG(preco) FROM produtos)
ORDER BY preco DESC
""")

In [ ]:
# De lista: clientes que compraram na categoria Notebooks
sql("""
SELECT id, nome, cidade, uf
FROM clientes
WHERE id IN (
    SELECT p.cliente_id
    FROM pedidos p
    JOIN itens_pedido i ON i.pedido_id = p.id
    JOIN produtos pr    ON pr.id = i.produto_id
    WHERE pr.categoria_id = 1
      AND p.status = 'pago'
)
ORDER BY nome
LIMIT 10
""")

In [ ]:
# Derivada (no FROM): agregue primeiro, depois filtre
sql("""
SELECT
    resumo.cidade,
    resumo.uf,
    resumo.pedidos,
    resumo.faturamento
FROM (
    SELECT
        c.cidade,
        c.uf,
        COUNT(DISTINCT p.id)                           AS pedidos,
        ROUND(SUM(i.quantidade * i.preco_unitario), 2) AS faturamento
    FROM pedidos p
    JOIN clientes c     ON c.id = p.cliente_id
    JOIN itens_pedido i ON i.pedido_id = p.id
    WHERE p.status = 'pago'
    GROUP BY c.cidade, c.uf
) AS resumo
WHERE resumo.faturamento > 40000
ORDER BY resumo.faturamento DESC
""")

In [ ]:
# Correlacionada: para CADA cliente, conta os pedidos dele
sql("""
SELECT
    c.nome,
    c.cidade,
    (SELECT COUNT(*)
     FROM pedidos p
     WHERE p.cliente_id = c.id) AS pedidos,
    (SELECT COUNT(*)
     FROM pedidos p
     WHERE p.cliente_id = c.id AND p.status = 'pago') AS pagos,
    (SELECT MAX(p.data_pedido)
     FROM pedidos p
     WHERE p.cliente_id = c.id) AS ultima_compra
FROM clientes c
ORDER BY pedidos DESC
LIMIT 10
""")

> ⚠️ **Subconsulta correlacionada é executada uma vez POR LINHA externa.** Com 30 clientes e 3 subconsultas, são 90 execuções. Com 300 mil clientes, é um problema sério.
>
> Quase sempre dá para reescrever com `LEFT JOIN` + `GROUP BY`, que o otimizador resolve em uma passada. Veja a comparação abaixo.

In [ ]:
# A mesma resposta, com LEFT JOIN — uma passada só
sql("""
SELECT
    c.nome,
    c.cidade,
    COUNT(p.id)                                              AS pedidos,
    SUM(CASE WHEN p.status = 'pago' THEN 1 ELSE 0 END)       AS pagos,
    MAX(p.data_pedido)                                       AS ultima_compra
FROM clientes c
LEFT JOIN pedidos p ON p.cliente_id = c.id
GROUP BY c.id, c.nome, c.cidade
ORDER BY pedidos DESC
LIMIT 10
""")

## 9. `EXISTS` e `NOT EXISTS`

`EXISTS` responde a uma pergunta booleana: *"existe pelo menos uma linha que satisfaça isto?"*

Ele **para na primeira linha encontrada** — não precisa contar tudo. Por isso costuma ser mais rápido que `IN` com subconsulta.

In [ ]:
sql("""
SELECT c.nome, c.cidade, c.uf, c.segmento
FROM clientes c
WHERE EXISTS (
    SELECT 1
    FROM pedidos p
    WHERE p.cliente_id = c.id
      AND p.status = 'pago'
)
ORDER BY c.nome
LIMIT 10
""")

> 💡 **Por que `SELECT 1`?** Porque o `EXISTS` só olha se **veio alguma linha** — o conteúdo é irrelevante. Escrever `SELECT 1` deixa isso explícito. `SELECT *` funcionaria igual; é só convenção.

In [ ]:
# NOT EXISTS: clientes sem NENHUM pedido pago
sql("""
SELECT c.nome, c.cidade, c.uf, c.data_cadastro
FROM clientes c
WHERE NOT EXISTS (
    SELECT 1
    FROM pedidos p
    WHERE p.cliente_id = c.id
      AND p.status = 'pago'
)
ORDER BY c.data_cadastro
""")

### 🔴 `NOT EXISTS` vs `NOT IN` — a diferença que salva relatórios

Lembra da aula 03_02: `x NOT IN (1, 2, NULL)` devolve `NULL`, e a linha é descartada.

Se a subconsulta de um `NOT IN` devolver **um único `NULL`**, a consulta inteira retorna **zero linhas** — sem erro, sem aviso. O `NOT EXISTS` não sofre disso.

In [ ]:
# Sabemos que existem 3 clientes sem nenhum pedido. As três consultas
# abaixo fazem a MESMA pergunta. Só duas acertam.

print("① NOT IN — funciona quando não há NULL:")
sql("""
SELECT COUNT(*) AS clientes_sem_pedido
FROM clientes c
WHERE c.id NOT IN (SELECT p.cliente_id FROM pedidos p)
""")

print("\n② NOT IN com UM único NULL na subconsulta:")
sql("""
SELECT COUNT(*) AS clientes_sem_pedido
FROM clientes c
WHERE c.id NOT IN (
    SELECT p.cliente_id FROM pedidos p
    UNION ALL
    SELECT NULL                       -- basta este para zerar tudo
)
""")

print("\n③ NOT EXISTS — imune ao problema:")
sql("""
SELECT COUNT(*) AS clientes_sem_pedido
FROM clientes c
WHERE NOT EXISTS (
    SELECT 1 FROM pedidos p WHERE p.cliente_id = c.id
)
""")

> 🧭 **Regra:** para "não está em", use **`NOT EXISTS`** ou **anti-join** (`LEFT JOIN ... WHERE ... IS NULL`). Reserve `NOT IN` para listas literais que você escreveu à mão.

## 10. CTEs — `WITH`, a ferramenta de legibilidade

Uma **CTE** (*Common Table Expression*) é uma consulta nomeada que existe só durante aquele comando.

```sql
WITH nome_da_cte AS (
    SELECT ...
),
outra_cte AS (
    SELECT ... FROM nome_da_cte ...
)
SELECT * FROM outra_cte;
```

**Por que isso muda sua vida:**

1. **Legibilidade.** Uma consulta com 3 subconsultas aninhadas é ilegível. As mesmas 3 como CTEs se leem de cima para baixo, como um script.
2. **Reuso.** A mesma CTE pode ser referenciada várias vezes.
3. **Depuração.** Comente o `SELECT` final e rode `SELECT * FROM primeira_cte` para inspecionar cada etapa.
4. **Recursão.** CTEs recursivas resolvem hierarquias (organograma, categorias aninhadas, explosão de materiais).

In [ ]:
# Compare: a mesma resposta em subconsulta aninhada...
sql("""
SELECT cidade, uf, faturamento,
       ROUND(100.0 * faturamento / (SELECT SUM(quantidade * preco_unitario)
                                    FROM itens_pedido i2
                                    JOIN pedidos p2 ON p2.id = i2.pedido_id
                                    WHERE p2.status = 'pago'), 1) AS share
FROM (
    SELECT c.cidade, c.uf,
           ROUND(SUM(i.quantidade * i.preco_unitario), 2) AS faturamento
    FROM pedidos p
    JOIN clientes c     ON c.id = p.cliente_id
    JOIN itens_pedido i ON i.pedido_id = p.id
    WHERE p.status = 'pago'
    GROUP BY c.cidade, c.uf
)
ORDER BY faturamento DESC
LIMIT 8
""")

In [ ]:
# ...e a mesma coisa com CTEs. Leia de cima para baixo.
sql("""
WITH vendas AS (
    SELECT
        c.cidade,
        c.uf,
        i.quantidade * i.preco_unitario AS valor
    FROM pedidos p
    JOIN clientes c     ON c.id = p.cliente_id
    JOIN itens_pedido i ON i.pedido_id = p.id
    WHERE p.status = 'pago'
),
por_cidade AS (
    SELECT cidade, uf, ROUND(SUM(valor), 2) AS faturamento
    FROM vendas
    GROUP BY cidade, uf
),
total AS (
    SELECT SUM(valor) AS geral FROM vendas
)
SELECT
    pc.cidade,
    pc.uf,
    pc.faturamento,
    ROUND(100.0 * pc.faturamento / t.geral, 1) AS share_pct
FROM por_cidade pc
CROSS JOIN total t
ORDER BY pc.faturamento DESC
LIMIT 8
""")

> 💭 **A segunda versão é maior em linhas e infinitamente menor em esforço de leitura.** Cada CTE tem um nome que explica o que ela é. Você entende a consulta na primeira passada — e daqui a seis meses também.
>
> **Esta é a habilidade que separa quem "sabe SQL" de quem escreve SQL que a equipe consegue manter.**

In [ ]:
# CTE encadeada: curva ABC de praças, calculada em etapas
sql("""
WITH vendas AS (
    SELECT c.cidade, i.quantidade * i.preco_unitario AS valor
    FROM pedidos p
    JOIN clientes c     ON c.id = p.cliente_id
    JOIN itens_pedido i ON i.pedido_id = p.id
    WHERE p.status = 'pago'
),
por_cidade AS (
    SELECT cidade, ROUND(SUM(valor), 2) AS faturamento
    FROM vendas GROUP BY cidade
),
com_share AS (
    SELECT
        cidade,
        faturamento,
        ROUND(100.0 * faturamento / (SELECT SUM(faturamento) FROM por_cidade), 2) AS share
    FROM por_cidade
),
acumulado AS (
    SELECT
        a.cidade,
        a.faturamento,
        a.share,
        ROUND((SELECT SUM(b.share) FROM com_share b
               WHERE b.faturamento >= a.faturamento), 2) AS share_acum
    FROM com_share a
)
SELECT
    cidade,
    faturamento,
    share,
    share_acum,
    CASE
        WHEN share_acum <= 80 THEN 'A'
        WHEN share_acum <= 95 THEN 'B'
        ELSE 'C'
    END AS classe
FROM acumulado
ORDER BY faturamento DESC
""")

### CTE recursiva

Uma CTE pode chamar a si mesma. A estrutura é sempre:

```sql
WITH RECURSIVE nome AS (
    SELECT ...            -- caso base (âncora)
    UNION ALL
    SELECT ... FROM nome  -- passo recursivo
)
```

Serve para hierarquias (organograma, categorias aninhadas), grafos e — o uso mais comum no dia a dia — **gerar séries**.

In [ ]:
# Gerando uma série de datas: essencial para relatórios sem buracos
sql("""
WITH RECURSIVE calendario(dia) AS (
    SELECT '2026-07-01'
    UNION ALL
    SELECT date(dia, '+1 day')
    FROM calendario
    WHERE dia < '2026-07-10'
)
SELECT
    c.dia,
    COUNT(DISTINCT p.id) AS pedidos,
    ROUND(COALESCE(SUM(i.quantidade * i.preco_unitario), 0), 2) AS faturamento
FROM calendario c
LEFT JOIN pedidos p      ON p.data_pedido = c.dia AND p.status = 'pago'
LEFT JOIN itens_pedido i ON i.pedido_id = p.id
GROUP BY c.dia
ORDER BY c.dia
""")

> 💡 **Por que gerar o calendário?** Porque um `GROUP BY data` só devolve dias em que **houve** venda. Se o dia 5 teve zero pedidos, ele simplesmente não aparece — e o gráfico da diretoria fica com um buraco em vez de um vale. O `LEFT JOIN` contra um calendário completo resolve isso.

In [ ]:
# Hierarquia: categorias com subcategorias
ddl("""
CREATE TABLE arvore_categorias (
    id       INTEGER PRIMARY KEY,
    nome     TEXT    NOT NULL,
    pai_id   INTEGER REFERENCES arvore_categorias(id)
);
INSERT INTO arvore_categorias VALUES
    (1, 'Informática',       NULL),
    (2, 'Computadores',      1),
    (3, 'Notebooks',         2),
    (4, 'Notebooks Gamer',   3),
    (5, 'Desktops',          2),
    (6, 'Periféricos',       1),
    (7, 'Mouses',            6),
    (8, 'Teclados',          6),
    (9, 'Teclados Mecânicos',8);
""")

sql("""
WITH RECURSIVE hierarquia(id, nome, nivel, caminho) AS (
    SELECT id, nome, 0, nome
    FROM arvore_categorias
    WHERE pai_id IS NULL

    UNION ALL

    SELECT
        f.id,
        f.nome,
        h.nivel + 1,
        h.caminho || ' > ' || f.nome
    FROM arvore_categorias f
    JOIN hierarquia h ON h.id = f.pai_id
)
SELECT
    substr('                    ', 1, nivel * 2) || nome AS estrutura,
    nivel,
    caminho
FROM hierarquia
ORDER BY caminho
""")

## 11. Operações de conjunto

| Operador | Faz | Remove duplicatas? |
|----------|-----|--------------------|
| `UNION` | Empilha dois resultados | ✅ Sim (custa mais) |
| `UNION ALL` | Empilha dois resultados | ❌ Não (**mais rápido**) |
| `INTERSECT` | Só o que está nos dois | ✅ |
| `EXCEPT` | O que está no primeiro e não no segundo | ✅ |

**Requisito:** os dois lados precisam ter o **mesmo número de colunas**, com tipos compatíveis. Os nomes vêm do primeiro `SELECT`.

> 💡 **Use `UNION ALL` por padrão.** Se você sabe que não há duplicatas, o `UNION` só desperdiça tempo ordenando para removê-las.

In [ ]:
sql("""
SELECT 'Produto caro'  AS tipo, nome, preco FROM produtos WHERE preco > 3000
UNION ALL
SELECT 'Produto barato', nome, preco FROM produtos WHERE preco < 100
ORDER BY preco DESC
""")

In [ ]:
# INTERSECT: clientes que compraram em Notebooks E em Periféricos
sql("""
SELECT p.cliente_id FROM pedidos p
JOIN itens_pedido i ON i.pedido_id = p.id
JOIN produtos pr    ON pr.id = i.produto_id
WHERE pr.categoria_id = 1 AND p.status = 'pago'

INTERSECT

SELECT p.cliente_id FROM pedidos p
JOIN itens_pedido i ON i.pedido_id = p.id
JOIN produtos pr    ON pr.id = i.produto_id
WHERE pr.categoria_id = 3 AND p.status = 'pago'
""")

In [ ]:
# EXCEPT: compraram Notebooks mas NUNCA Periféricos
sql("""
WITH compradores_notebook AS (
    SELECT DISTINCT p.cliente_id FROM pedidos p
    JOIN itens_pedido i ON i.pedido_id = p.id
    JOIN produtos pr    ON pr.id = i.produto_id
    WHERE pr.categoria_id = 1 AND p.status = 'pago'
),
compradores_periferico AS (
    SELECT DISTINCT p.cliente_id FROM pedidos p
    JOIN itens_pedido i ON i.pedido_id = p.id
    JOIN produtos pr    ON pr.id = i.produto_id
    WHERE pr.categoria_id = 3 AND p.status = 'pago'
),
alvo AS (
    SELECT cliente_id FROM compradores_notebook
    EXCEPT
    SELECT cliente_id FROM compradores_periferico
)
SELECT c.nome, c.cidade, c.uf
FROM alvo a
JOIN clientes c ON c.id = a.cliente_id
ORDER BY c.nome
""")

> 💼 **Repare no que essa última consulta é:** uma lista de campanha. "Clientes que compraram notebook e nunca levaram um periférico" é exatamente o público para uma oferta de mouse e teclado. SQL não é só relatório — é ferramenta de decisão.

## 🔧 Prática guiada — Perguntas de negócio de verdade

In [ ]:
# ── 1. Quem são os melhores clientes? (RFM simplificado) ─────
sql("""
WITH compras AS (
    SELECT
        p.cliente_id,
        p.id                              AS pedido_id,
        p.data_pedido,
        SUM(i.quantidade * i.preco_unitario) AS valor
    FROM pedidos p
    JOIN itens_pedido i ON i.pedido_id = p.id
    WHERE p.status = 'pago'
    GROUP BY p.id, p.cliente_id, p.data_pedido
),
metricas AS (
    SELECT
        cliente_id,
        COUNT(*)                                       AS frequencia,
        ROUND(SUM(valor), 2)                           AS monetario,
        ROUND(AVG(valor), 2)                           AS ticket_medio,
        MAX(data_pedido)                               AS ultima_compra,
        CAST(julianday('2026-08-01') - julianday(MAX(data_pedido)) AS INTEGER) AS dias_sem_comprar
    FROM compras
    GROUP BY cliente_id
)
SELECT
    c.nome,
    c.cidade,
    c.segmento,
    m.frequencia,
    m.monetario,
    m.ticket_medio,
    m.dias_sem_comprar,
    CASE
        WHEN m.frequencia >= 8 AND m.dias_sem_comprar <= 30 THEN 'Campeão'
        WHEN m.frequencia >= 5                              THEN 'Fiel'
        WHEN m.dias_sem_comprar > 60                        THEN 'Em risco'
        ELSE 'Regular'
    END AS classificacao
FROM metricas m
JOIN clientes c ON c.id = m.cliente_id
ORDER BY m.monetario DESC
LIMIT 12
""")

In [ ]:
# ── 2. Quais produtos são comprados juntos? (market basket) ──
sql("""
WITH pares AS (
    SELECT
        a.produto_id AS produto_a,
        b.produto_id AS produto_b,
        COUNT(*)     AS vezes_juntos
    FROM itens_pedido a
    JOIN itens_pedido b ON b.pedido_id = a.pedido_id
                        AND b.produto_id > a.produto_id
    JOIN pedidos p      ON p.id = a.pedido_id
    WHERE p.status = 'pago'
    GROUP BY a.produto_id, b.produto_id
    HAVING COUNT(*) >= 2
)
SELECT
    pa.nome  AS produto_a,
    pb.nome  AS produto_b,
    pr.vezes_juntos
FROM pares pr
JOIN produtos pa ON pa.id = pr.produto_a
JOIN produtos pb ON pb.id = pr.produto_b
ORDER BY pr.vezes_juntos DESC, pa.nome
LIMIT 12
""")

In [ ]:
# ── 3. Desempenho por categoria, com produtos parados ────────
sql("""
WITH vendas_produto AS (
    SELECT
        pr.id,
        pr.categoria_id,
        COALESCE(SUM(i.quantidade), 0)                    AS unidades,
        COALESCE(SUM(i.quantidade * i.preco_unitario), 0) AS receita
    FROM produtos pr
    LEFT JOIN itens_pedido i ON i.produto_id = pr.id
    LEFT JOIN pedidos p      ON p.id = i.pedido_id AND p.status = 'pago'
    GROUP BY pr.id, pr.categoria_id
)
SELECT
    cat.nome                                     AS categoria,
    COUNT(*)                                     AS produtos,
    SUM(CASE WHEN vp.unidades = 0 THEN 1 ELSE 0 END) AS sem_giro,
    SUM(vp.unidades)                             AS unidades,
    ROUND(SUM(vp.receita), 2)                    AS receita,
    ROUND(100.0 * SUM(vp.receita)
          / (SELECT SUM(receita) FROM vendas_produto), 1) AS share_pct
FROM vendas_produto vp
JOIN categorias cat ON cat.id = vp.categoria_id
GROUP BY cat.id, cat.nome
ORDER BY receita DESC
""")

In [ ]:
# ── 4. Funil: cadastrou → comprou → recomprou ────────────────
sql("""
WITH atividade AS (
    SELECT
        c.id,
        COUNT(DISTINCT p.id)                                        AS pedidos,
        COUNT(DISTINCT CASE WHEN p.status='pago' THEN p.id END)     AS pagos
    FROM clientes c
    LEFT JOIN pedidos p ON p.cliente_id = c.id
    GROUP BY c.id
)
SELECT
    'Cadastrados'          AS etapa, COUNT(*) AS clientes FROM atividade
UNION ALL SELECT
    'Fizeram 1+ pedido',   COUNT(*) FROM atividade WHERE pedidos >= 1
UNION ALL SELECT
    'Tiveram 1+ pago',     COUNT(*) FROM atividade WHERE pagos >= 1
UNION ALL SELECT
    'Recompraram (2+)',    COUNT(*) FROM atividade WHERE pagos >= 2
UNION ALL SELECT
    'Fiéis (5+)',          COUNT(*) FROM atividade WHERE pagos >= 5
""")

## 📝 Exercícios

**E1.** Liste todos os pedidos com o nome do cliente, a cidade e o valor total do pedido (soma dos itens). Ordene do maior para o menor.

**E2.** Quais categorias **não têm nenhum produto** com estoque acima de 50? Use anti-join ou `NOT EXISTS`.

**E3.** Para cada cliente, mostre nome, total de pedidos, pedidos pagos e valor total gasto. **Inclua os clientes sem pedido** (com zeros).

**E4.** Liste os produtos que foram vendidos em **todos os três canais** (`site`, `app`, `marketplace`). *(Dica: `GROUP BY` + `HAVING COUNT(DISTINCT canal) = 3`.)*

**E5.** Reescreva esta subconsulta correlacionada usando `LEFT JOIN` + `GROUP BY`:
```sql
SELECT c.nome,
       (SELECT COUNT(*) FROM pedidos p WHERE p.cliente_id = c.id) AS n
FROM clientes c;
```

**E6.** Usando CTEs, calcule o faturamento por mês e a **variação percentual** em relação ao mês anterior. *(Dica: faça um self join da CTE mensal consigo mesma.)*

**E7.** Encontre os pares de clientes que compraram **o mesmo produto**. Mostre o produto e os dois nomes, sem repetir pares.

**E8.** Gere um calendário recursivo de todos os dias de julho/2026 e mostre, para cada dia, o faturamento (zero nos dias sem venda).

**E9.** Quais clientes compraram produtos de **mais de 3 categorias distintas**? Mostre o nome e a lista de categorias com `GROUP_CONCAT`.

**E10.** Monte um relatório com uma linha por categoria contendo: nº de produtos, nº de produtos vendidos, nº sem giro, receita total e o nome do produto campeão da categoria.

In [ ]:
# E1

In [ ]:
# E2

In [ ]:
# E3

In [ ]:
# E4

In [ ]:
# E5

In [ ]:
# E6

In [ ]:
# E7

In [ ]:
# E8

In [ ]:
# E9

In [ ]:
# E10

## 📋 Cola de referência

```sql
-- ── Tipos de JOIN ──
FROM a JOIN  b ON b.a_id = a.id      -- INNER: só o que casa nos dois
FROM a LEFT  JOIN b ON ...           -- tudo de A, NULL onde não casar
FROM a RIGHT JOIN b ON ...           -- tudo de B (SQLite 3.39+; prefira inverter e usar LEFT)
FROM a FULL  OUTER JOIN b ON ...     -- tudo dos dois lados
FROM a CROSS JOIN b                  -- todas as combinações

-- ── Anti-join: quem NÃO tem correspondência ──
FROM a LEFT JOIN b ON b.a_id = a.id
WHERE b.id IS NULL

-- ── Self join: combinações dois a dois ──
FROM t x JOIN t y ON y.grupo = x.grupo AND y.id > x.id

-- ⚠️ LEFT JOIN: condição da DIREITA vai no ON, não no WHERE
LEFT JOIN pedidos p ON p.id = i.pedido_id AND p.status = 'pago'   -- ✅
LEFT JOIN pedidos p ON p.id = i.pedido_id WHERE p.status = 'pago' -- ❌ vira INNER

-- ⚠️ JOIN multiplica linhas
COUNT(DISTINCT p.id)     -- conta pedidos, não linhas
COUNT(i.id)              -- em LEFT JOIN, conta 0 corretamente (COUNT(*) contaria 1)

-- ── Subconsultas ──
WHERE preco > (SELECT AVG(preco) FROM produtos)      -- escalar
WHERE id IN (SELECT cliente_id FROM pedidos)         -- de lista
FROM (SELECT ... GROUP BY ...) AS resumo             -- derivada
WHERE EXISTS (SELECT 1 FROM p WHERE p.c_id = c.id)   -- correlacionada

-- ⚠️ Para "não está em", use NOT EXISTS (imune a NULL), nunca NOT IN

-- ── CTE ──
WITH etapa1 AS (
    SELECT ...
),
etapa2 AS (
    SELECT ... FROM etapa1
)
SELECT * FROM etapa2;

-- ── CTE recursiva ──
WITH RECURSIVE serie(n) AS (
    SELECT 1                       -- âncora
    UNION ALL
    SELECT n + 1 FROM serie WHERE n < 10   -- passo (com PARADA!)
)
SELECT * FROM serie;

WITH RECURSIVE cal(dia) AS (
    SELECT '2026-07-01'
    UNION ALL
    SELECT date(dia, '+1 day') FROM cal WHERE dia < '2026-07-31'
)

-- ── Conjuntos ──
SELECT ... UNION ALL SELECT ...    -- empilha (rápido, mantém duplicatas)
SELECT ... UNION     SELECT ...    -- empilha e deduplica
SELECT ... INTERSECT SELECT ...    -- só o que está nos dois
SELECT ... EXCEPT    SELECT ...    -- o que está no 1º e não no 2º
```

## ✅ Checklist de saída

- [ ] Escrevo `JOIN ... ON` com apelidos de tabela significativos
- [ ] Encadeio vários `JOIN` seguindo o caminho do modelo
- [ ] **Sei que o `JOIN` multiplica linhas** e uso `COUNT(DISTINCT ...)`
- [ ] Sei que somar colunas do "pai" após juntar com o "filho" infla o total
- [ ] Escolho `LEFT JOIN` quando os zeros importam
- [ ] Uso `COUNT(coluna)` em vez de `COUNT(*)` com `LEFT JOIN`
- [ ] Coloco condições da tabela da direita no `ON`, não no `WHERE`
- [ ] Escrevo anti-join (`LEFT JOIN ... WHERE ... IS NULL`) para achar órfãos
- [ ] Sei que `RIGHT JOIN` é dispensável — basta inverter as tabelas
- [ ] Reconheço um `CROSS JOIN` acidental
- [ ] Faço self join com `b.id > a.id` para combinações
- [ ] Diferencio subconsulta escalar, de lista, derivada e correlacionada
- [ ] Sei que correlacionada roda uma vez por linha e sei reescrevê-la
- [ ] Uso `NOT EXISTS` em vez de `NOT IN` para negação
- [ ] **Escrevo CTEs em vez de subconsultas aninhadas**
- [ ] Sei usar `WITH RECURSIVE` para séries e hierarquias
- [ ] Prefiro `UNION ALL` quando não preciso deduplicar

---

### ➡️ Próxima aula

**`03_04_Manutencao_e_Transacoes.ipynb`** — `INSERT`, `UPDATE`, `DELETE`, índices e transações. Onde você para de só ler e começa a **escrever** no banco com segurança.